# ShellWhisperer-1.5B — Unsloth Fine-Tuning (Optimized)

Fine-tune `Qwen/Qwen2.5-Coder-1.5B-Instruct` on shell/command-line data.

**Platform**: Kaggle T4 (30h/week free) or Colab T4 (12h free)
**Time**: ~15-20 min | **VRAM**: ~6GB with 4-bit QLoRA

### Setup Checklist
- [ ] Kaggle: Settings → Accelerator → GPU T4, Internet → On
- [ ] Colab: Runtime → Change runtime → T4 GPU
- [ ] Upload training data files to session storage or Google Drive

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps --force-reinstall git+https://github.com/unslothai/unsloth.git
!pip install --no-deps trl==0.15.2

from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

In [ ]:
# ════════════════════════════════════════════════════════
# Configuration
# ════════════════════════════════════════════════════════

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
HF_REPO = "fableforge-ai/ShellWhisperer-1.5B"  # Upload target
MAX_SEQ_LENGTH = 2048

# LoRA — rank 16 for better capacity than our r=8 MPS run
LORA_R = 16
LORA_ALPHA = 16

# Training — 5 epochs for small dataset (134 examples)
NUM_EPOCHS = 5
BATCH_SIZE = 2
GRAD_ACCUM = 4        # Effective batch = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10

print(f"Config: r={LORA_R}, epochs={NUM_EPOCHS}, lr={LEARNING_RATE}, effective_batch={BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# ════════════════════════════════════════════════════════
# Load model with 4-bit QLoRA
# ════════════════════════════════════════════════════════

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

In [ ]:
# ════════════════════════════════════════════════════════
# Load and format training data
# ════════════════════════════════════════════════════════

from datasets import Dataset
import json

# Option 1: From local files (upload to session storage)
TRAIN_PATH = "shellwhisperer_train.jsonl"
VAL_PATH = "shellwhisperer_val.jsonl"

# Option 2: From HuggingFace (after uploading)
# from datasets import load_dataset
# ds = load_dataset("fableforge-ai/shellwhisperer-data")

SYSTEM_PROMPT = "You are ShellWhisperer, an expert in command-line tools, shell scripting, and system administration. Provide concise, accurate commands with brief explanations."

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def to_chatml(example):
    if "messages" in example:
        return example  # Already ChatML
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["instruction"] + ("\n" + example["input"] if example.get("input") else "")},
            {"role": "assistant", "content": example["output"]},
        ]
    }

raw_train = load_jsonl(TRAIN_PATH)
raw_val = load_jsonl(VAL_PATH)

train_data = [to_chatml(ex) for ex in raw_train]
val_data = [to_chatml(ex) for ex in raw_val]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Format to text
def format_to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_to_text)
val_dataset = val_dataset.map(format_to_text)
print(f"Formatted. Sample: {train_dataset[0]['text'][:200]}...")

In [ ]:
# ════════════════════════════════════════════════════════
# Train!
# ════════════════════════════════════════════════════════

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_steps=50,
        save_total_limit=3,
        eval_strategy="steps",
        eval_steps=50,
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining complete! Time: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ════════════════════════════════════════════════════════
# Quick inference test
# ════════════════════════════════════════════════════════

FastLanguageModel.for_inference(model)

test_prompts = [
    "Find all Python files modified in the last week",
    "Kill a process listening on port 3000",
    "Show disk usage sorted by size for /home",
    "Check SSL certificate expiration for example.com",
    "Create a temporary directory and clean up on exit",
    "How do I recursively find and replace text in files?",
    "Monitor a log file in real-time filtering for ERROR",
    "Show all SUID-root binaries on the system",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.3, top_p=0.9, use_cache=True)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {prompt}")
    print(f"A: {response}\n")

In [ ]:
# ════════════════════════════════════════════════════════
# Save LoRA adapters
# ════════════════════════════════════════════════════════

model.save_pretrained("shellwhisperer-lora")
tokenizer.save_pretrained("shellwhisperer-lora")
print("LoRA adapters saved to ./shellwhisperer-lora")

In [ ]:
# ════════════════════════════════════════════════════════
# Merge LoRA into base model (standalone 16-bit)
# ════════════════════════════════════════════════════════

model.save_pretrained_merged("shellwhisperer-merged", tokenizer)
print("Merged model saved to ./shellwhisperer-merged")

In [ ]:
# ════════════════════════════════════════════════════════
# Export as GGUF for Ollama/llama.cpp
# ════════════════════════════════════════════════════════

model.save_pretrained_gguf("shellwhisperer-gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF model saved to ./shellwhisperer-gguf")
print("Use with: ollama create shellwhisperer -f Modelfile")

In [ ]:
# ════════════════════════════════════════════════════════
# Push to HuggingFace
# ════════════════════════════════════════════════════════

# Uncomment and add your token:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

# Push merged model (recommended)
# model.push_to_hub_merged(HF_REPO, tokenizer)

# Or push LoRA adapters only (smaller upload)
# model.push_to_hub(HF_REPO + "-LoRA")
# tokenizer.push_to_hub(HF_REPO + "-LoRA")

# Push GGUF
# model.push_to_hub_gguf(HF_REPO + "-GGUF", tokenizer, quantization_method="q4_k_m")

print(f"Target: https://huggingface.co/{HF_REPO}")
print("Uncomment the push lines above after logging in.")